# PySpark 3: types, functions and nulls

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-03-types-functions-and-nulls.ipynb)

A vendor file has amounts like `"1,204.50"`, dates like `"03/01/2026"`, missing regions, emails in mixed case,
and one amount of `"N/A"`. Before any join or report, every value needs the right type and a deliberate
answer for "what if it is missing or wrong?".

This lesson covers Spark's data types, casting under Spark 4's ANSI mode, how nulls behave, conditional logic,
and the built-in string, number, date and hashing functions you will use in almost every pipeline.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 2: DataFrame essentials](pyspark-02-dataframe-essentials.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered) and `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F
from decimal import Decimal

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

## Choose the right data type

| type | holds | use for |
| --- | --- | --- |
| `int`, `bigint` | 32- and 64-bit integers | ids, counts |
| `double` | binary floating point | measurements, scores, ratios |
| `decimal(p, s)` | exact numbers with `p` digits, `s` after the point | **money**, quantities that must add up |
| `string` | text | names, codes |
| `boolean` | true / false | flags |
| `date` | a calendar day | business dates, partitions |
| `timestamp` | an instant, shown in the session time zone | event times |
| `timestamp_ntz` | a wall-clock time with no time zone | store opening hours, local schedules |
| `array`, `struct`, `map` | nested values | covered in [lesson 8](pyspark-08-nested-and-semi-structured-data.ipynb) |

A `double` stores numbers in binary, so most decimal fractions are approximations; sums of many of them drift.
A `decimal` stores them exactly, at the cost of a fixed number of digits.

**Where it matters**
- **Finance tables:** revenue reconciliations that must match to the cent.
- **Schemas for consumers:** downstream tools read your types; `bigint` vs `int` or `decimal` vs `double` is part of the contract.

**Watch out for**
- A decimal result can exceed its precision: Spark widens precision for arithmetic, up to 38 digits.
- Using `string` for everything postpones type errors to the consumer.

### Example

Add 0.1 and 0.2 as `double` and as `decimal`, and compare each result with 0.3.

In [ ]:
prices = spark.createDataFrame([(0.1, 0.2, Decimal("0.1"), Decimal("0.2"))], "a double, b double, c decimal(4,2), d decimal(4,2)")
result = prices.select(
    (F.col("a") + F.col("b")).alias("double_sum"),
    (F.col("a") + F.col("b") == 0.3).alias("double_equals_0_3"),
    (F.col("c") + F.col("d")).alias("decimal_sum"),
    (F.col("c") + F.col("d") == Decimal("0.3")).alias("decimal_equals_0_3"),
)
result.show()
result.printSchema()

**What to notice:** As doubles, 0.1 + 0.2 is `0.30000000000000004`, so it is not equal to 0.3: neither 0.1 nor 0.2 can be stored
exactly in binary. As decimals the sum is exactly `0.30`. The schema shows Spark widened the decimal result to
`decimal(5,2)` so the addition cannot overflow. Errors like this one, repeated over millions of rows, are why
revenue reconciliations fail on `double` columns.

## Cast types and handle bad values

`F.col("x").cast("int")` converts a column. Spark 4 runs in **ANSI SQL mode** by default: a value that cannot be
converted (`"N/A"` to `int`), an integer overflow, or a division by zero **raises an error** instead of quietly
returning null. When bad values are expected and should be handled, use the `try_` functions, which return
null instead: `try_cast` (via `F.expr`), `F.try_divide`, `F.try_add`, `F.try_to_number`, `F.try_to_timestamp`.

**Where it matters**
- **Typing raw strings** from CSV or JSON into numbers and dates in a silver table.
- **Upgrades from Spark 3,** where the same casts silently produced nulls.

**Watch out for**
- A `try_` null is still a problem to count: compare raw and typed columns to find rejected values.
- Formatted numbers such as `"1,204.50"` need `try_to_number` with a format, not `cast`.

### Example

Cast clean and dirty strings, see the error from a plain cast, and handle the same values with `try_` functions.

In [ ]:
raw = spark.createDataFrame(
    [("o1", "120.50", "4"), ("o2", "N/A", "0"), ("o3", "1,204.50", "2")],
    "order_id string, amount_text string, units_text string",
)

spark.sparkContext.setLogLevel("OFF")  # the failure below is expected
try:
    raw.select(F.col("amount_text").cast("decimal(12,2)")).collect()
except Exception as exc:
    print("plain cast:", str(exc).splitlines()[0][:90])
finally:
    spark.sparkContext.setLogLevel("ERROR")

typed = raw.select(
    "order_id",
    "amount_text",
    F.expr("try_cast(amount_text AS DECIMAL(12,2))").alias("try_cast"),
    F.try_to_number("amount_text", F.lit("9,999.99")).alias("try_to_number"),
    F.try_divide(F.lit(100), F.col("units_text").cast("int")).alias("per_unit"),
)
typed.show()

**What to notice:** The plain cast failed on the whole job because of one value. `try_cast` returned null for both `N/A` and
`1,204.50`, since the comma is not part of a plain number. `try_to_number` with the format `9,999.99` parsed
`1,204.50` correctly and still returned null for `N/A`. `try_divide` returned null for the division by zero
instead of failing.

## Handle nulls

Null means "unknown". Any comparison or arithmetic with null gives null, and a filter keeps only rows where the
condition is **true**, so null rows silently disappear. Tools for nulls:

- `isNull()` / `isNotNull()` to test; `eqNullSafe` (SQL `<=>`) to compare where null equals null;
- `F.coalesce(a, b, ...)` for the first non-null value; `fillna` (`na.fill`) to replace nulls per column;
- `dropna` (`na.drop`) to remove rows with nulls in chosen columns; `F.nullif(a, b)` to turn a sentinel into null.

Aggregates skip nulls: `count(col)` counts non-null values while `count("*")` counts rows, and `avg` divides by
the non-null count.

**Where it matters**
- **Defaults:** an unknown region becomes `unknown` so reports do not lose the row.
- **Sentinels:** source systems that write `''` or `-1` for missing values.

**Watch out for**
- Filling a numeric null with 0 changes averages and sums; fill only when missing really means zero.
- `col != 'x'` drops null rows. Decide explicitly whether they should stay.

### Example

Compare with and without null safety, fill and drop nulls, turn an empty string into null, and count with and without nulls.

In [ ]:
contacts = spark.createDataFrame(
    [(1, "North", "a@x.com"), (2, None, ""), (3, "South", None), (4, None, "d@x.com")],
    "id int, region string, email string",
)
contacts.select(
    "id", "region",
    (F.col("region") == F.lit(None)).alias("equals_null"),
    F.col("region").eqNullSafe(F.lit(None)).alias("null_safe"),
    F.coalesce("region", F.lit("unknown")).alias("region_or_default"),
    F.nullif("email", F.lit("")).alias("email_clean"),
).show()

print("dropna(subset=['region']):", contacts.dropna(subset=["region"]).count(), "rows")
contacts.fillna({"region": "unknown", "email": "none"}).show()
contacts.agg(F.count("*").alias("rows"), F.count("region").alias("with_region")).show()

**What to notice:** `region == null` is null for every row, even where region is null, so it can never select anything; only
`eqNullSafe` returns true for the missing regions. `nullif` turned customer 2's empty email into a real null.
`count("*")` counts 4 rows but `count("region")` only the 2 known regions.

## Write conditional logic

`F.when(condition, value).when(...).otherwise(default)` is SQL's `CASE WHEN`. Conditions are checked in order
and the first true one wins. Without `otherwise`, unmatched rows get null. `F.greatest` and `F.least` pick the
largest or smallest of several columns, ignoring nulls.

**Where it matters**
- **Business tiers and flags:** order size bands, `is_completed`, risk levels.
- **Cleaning:** map many source spellings to one value.

**Watch out for**
- Test the largest threshold first; `amount >= 100` before `amount >= 1000` puts every large order in the wrong tier.
- A missing `otherwise` produces nulls that later filters silently drop.

### Example

Assign size tiers with `when`, and show what happens to rows that match no branch when `otherwise` is missing.

In [ ]:
orders.select(
    "order_id", "status", "amount",
    F.when(F.col("amount") >= 1000, "enterprise")
     .when(F.col("amount") >= 100, "mid_market")
     .otherwise("small").alias("tier"),
    F.when(F.col("status").isin("paid", "shipped"), True).alias("revenue_flag"),
).orderBy("order_id").show()

**What to notice:** Only order 9 is `enterprise`, because the `>= 1000` branch is tested first. `revenue_flag` has no `otherwise`, so
placed and returned orders get null instead of false; a later `where(~revenue_flag)` would miss them.

## Work with strings

The most used string functions: `trim`, `lower`, `upper`, `initcap`; `concat` and `concat_ws` (with a
separator, skipping nulls); `substring`, `length`, `lpad`, `rpad`; `split` into an array; `regexp_extract` to
pull out a pattern group, `regexp_replace` to rewrite it; `translate` for character-by-character replacement;
`format_string` for printf-style formatting.

**Where it matters**
- **Join keys:** `" Ada@Example.COM "` must become `"ada@example.com"` before it can match.
- **Parsing codes:** pull a region or a numeric id out of a compound string.

**Watch out for**
- `concat` returns null if any input is null; `concat_ws` skips nulls.
- `substring` positions start at 1, not 0.
- `regexp_extract` returns an empty string, not null, when the pattern does not match.

### Example

Normalize emails, build a display name, pad an id, extract parts with regular expressions, and split a code.

In [ ]:
people = spark.createDataFrame(
    [("  Ada@Example.COM ", "ada", "lovelace", 7, "UK-LON-0042"),
     ("grace@example.org", "grace", None, 42, "US-NYC-0007")],
    "email string, first string, last string, id int, location string",
)
people.select(
    F.lower(F.trim("email")).alias("email"),
    F.concat(F.initcap("first"), F.lit(" "), F.initcap("last")).alias("concat"),
    F.concat_ws(" ", F.initcap("first"), F.initcap("last")).alias("concat_ws"),
    F.lpad(F.col("id").cast("string"), 5, "0").alias("padded_id"),
    F.regexp_extract("location", r"^([A-Z]{2})-([A-Z]{3})-(\d+)$", 2).alias("city"),
    F.regexp_replace("location", r"\d", "#").alias("masked"),
    F.split("location", "-").alias("parts"),
).show(truncate=False)

**What to notice:** Grace has no last name: `concat` returned null for her, while `concat_ws` skipped the null and returned
`Grace`. `regexp_extract` with group 2 pulled out the city code, and `split` returned an array you can index
later. `lpad` made every id five characters wide.

## Round and compute numbers

`round(col, n)` rounds half up (2.5 becomes 3); `bround` rounds half to even (2.5 becomes 2), which avoids a
bias in large sums. `ceil`, `floor`, `abs` and `pow` do what their names say. Keep money in `decimal` and
round only for display or at a defined business step, not at every intermediate calculation.

**Where it matters**
- **Financial reporting rules,** which often specify the rounding mode.
- **Percentages and ratios** computed from integer counts.

**Watch out for**
- Integer division does not truncate in Spark: `7 / 2` is `3.5`. Use `F.floor` or `div` for whole numbers.
- Rounding before summing gives a different total than summing before rounding.

### Example

Compare `round` and `bround` on halves, and divide integers.

In [ ]:
halves = spark.createDataFrame([(Decimal("0.5"),), (Decimal("1.5"),), (Decimal("2.5"),)], "value decimal(3,1)")
halves.select(
    "value", F.round("value").alias("round"), F.bround("value").alias("bround"),
    F.ceil("value").alias("ceil"), F.floor("value").alias("floor"),
).show()
spark.range(1).select((F.lit(7) / 2).alias("7 / 2"), F.expr("7 div 2").alias("7 div 2")).show()

**What to notice:** `round` sends every half upward (1, 2, 3); `bround` sends halves to the nearest even number (0, 2, 2), so
rounding errors cancel out over many values. `7 / 2` is `3.5`, a decimal division; `div` gives the integer 3.

## Work with dates and timestamps

Parse with `to_date` and `to_timestamp`, giving a format such as `"dd/MM/yyyy"` when the input is not ISO 8601.
Format back to text with `date_format`. Do arithmetic with `date_add`, `date_sub`, `add_months`, `datediff`,
`months_between`; truncate with `date_trunc("month", ts)` or `trunc(date, "month")`; extract parts with `year`,
`month`, `dayofweek`, `weekofyear`; and build dates with `make_date`.

A `timestamp` is an instant, displayed in the session time zone (`spark.sql.session.timeZone`).
`from_utc_timestamp` and `to_utc_timestamp` convert between UTC and a named zone.

**Where it matters**
- **Business dates:** an order at 00:30 UTC belongs to the previous day in Chicago.
- **Partitions and reports** by day, week or month.

**Watch out for**
- Format letters are case-sensitive: `MM` is month, `mm` is minute; `yyyy` is year.
- Set the session time zone explicitly in every job; a cluster default can shift every derived date.
- `current_date()` makes results depend on when the job ran, which breaks reruns. Pass the business date in.

### Example

Parse European-style dates, derive calendar fields, and convert order 12 from UTC to Chicago time.

In [ ]:
deliveries = spark.createDataFrame([("03/01/2026", "2026-01-31 18:45")], "day_text string, ts_text string")
deliveries.select(
    F.to_date("day_text", "dd/MM/yyyy").alias("day"),
    F.to_timestamp("ts_text", "yyyy-MM-dd HH:mm").alias("ts"),
    F.date_format(F.to_date("day_text", "dd/MM/yyyy"), "EEEE d MMMM yyyy").alias("pretty"),
    F.last_day(F.to_date("day_text", "dd/MM/yyyy")).alias("month_end"),
    F.add_months(F.to_date("day_text", "dd/MM/yyyy"), 1).alias("next_month"),
).show(truncate=False)

orders.where("order_id in (11, 12)").select(
    "order_id", "order_ts",
    F.to_date("order_ts").alias("utc_date"),
    F.from_utc_timestamp("order_ts", "America/Chicago").alias("chicago_time"),
    F.to_date(F.from_utc_timestamp("order_ts", "America/Chicago")).alias("chicago_date"),
    F.dayofweek("order_ts").alias("day_of_week"),
    F.datediff(F.lit("2026-01-31").cast("date"), F.to_date("order_ts")).alias("days_to_month_end"),
).show()

**What to notice:** `03/01/2026` became 3 January because the format says day first. Order 12 was placed at 00:30 UTC on 14 January,
which is still 13 January in Chicago, so the two business dates differ. `dayofweek` counts from Sunday = 1.

## Generate ids and hashes

- `sha2(col, 256)` and `md5` produce stable hashes of a value: the same input always gives the same output.
  `xxhash64` is a fast 64-bit hash. Hash several columns with `concat_ws` first.
- `monotonically_increasing_id()` gives each row a unique 64-bit number, but **not** consecutive or stable:
  the values encode the partition, and change if the data is repartitioned.
- `F.uuid()` gives a random identifier per row; it changes every time the plan runs.

**Where it matters**
- **Surrogate keys and change detection:** hash the business key, or hash all attributes to spot changed rows.
- **Pseudonymizing** emails before sharing a dataset.

**Watch out for**
- Never use `monotonically_increasing_id` or `uuid` as a key you need to reproduce on a rerun.
- Hashing `concat(a, b)` makes `('ab', 'c')` and `('a', 'bc')` collide; use a separator with `concat_ws`.

### Example

Hash emails and a row's attributes, and see what `monotonically_increasing_id` produces across partitions.

In [ ]:
customers.select(
    "customer_id",
    F.sha2(F.lower("name"), 256).substr(1, 16).alias("name_sha256"),
    F.xxhash64("customer_id", "name", "region").alias("row_hash"),
).show(3, truncate=False)

spark.range(6, numPartitions=3).select("id", F.monotonically_increasing_id().alias("mono_id")).show()

**What to notice:** The hashes are deterministic: rerun the cell and they stay the same. The `mono_id` values jump between
partitions (0, 1, then 8589934592, ...) because the upper bits hold the partition number. They are unique but
not consecutive, and a different partitioning would give different numbers.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Why store money as `decimal` rather than `double`?

**A.** A `double` stores binary approximations, so sums drift (ten times 0.1 is 0.9999999999999999). A `decimal` is exact.

**Q2.** In Spark 4, what does casting `"N/A"` to `int` do, and how do you get a null instead?

**A.** ANSI mode is on, so the cast raises `CAST_INVALID_INPUT` and the job fails. `try_cast` returns null.

**Q3.** Why does `region == None` never select anything?

**A.** Any comparison with null gives null, and a filter keeps only true rows. Use `isNull()` or `eqNullSafe`.

**Q4.** What is the difference between `count('*')` and `count('region')`?

**A.** `count('*')` counts rows; `count('region')` counts non-null values of `region`.

**Q5.** Why might `concat` return null where `concat_ws` does not?

**A.** `concat` returns null if any input is null; `concat_ws` skips nulls.

**Q6.** Can you use `monotonically_increasing_id` as a primary key?

**A.** Only within one run. The values depend on partitioning, are not consecutive, and change on a rerun. Hash a business key instead.

## Key takeaways

- Money is `decimal`; measurements can be `double`.
- Spark 4 casts fail loudly; use `try_` functions where bad values are expected, then count the nulls.
- Nulls fail comparisons and are skipped by aggregates: handle them on purpose.
- Order `when` branches from the most specific; always decide what `otherwise` should be.
- Give date formats explicitly and pin the session time zone.
- Hash business keys for reproducible ids.

**Next:** Practise this module in [PySpark drills 3: types, functions and nulls](pyspark-drills-03-types-functions-and-nulls.ipynb), then continue with [PySpark 4: reading and writing data](pyspark-04-reading-and-writing-data.ipynb).